# MCP Evaluation Benchmark — Multi-Server

Generate a synthetic evaluation benchmark for MCP tool-use skills across multiple servers, then validate
that it produces the same model rankings as [Accenture's mcp-bench](https://github.com/Accenture/mcp-bench).

### Pipeline

```
MCP Servers ──> Distillation Flow ──> Synthetic Tasks + Expert Trajectories ──> Model Evaluation ──> Rankings
```

### Servers (6 data-dependent)

| Server | Default Port | Tools | Data type |
|--------|-------------|-------|-----------| 
| Weather Data | 8001 | 4 | Live weather API |
| Medical Calculator | 8002 | 22 | Clinical formulas |
| Wikipedia | 8003 | 9 | Live article content |
| Car Price Evaluator | 8004 | 3 | Vehicle pricing DB |
| Reddit | 8005 | 2 | Live posts/comments |
| DEX Paprika | 8006 | 11 | DeFi/crypto market data |

---
## 0. Setup

### 0.1 Prerequisites

**MCP servers**: Clone [mcp-bench](https://github.com/Accenture/mcp-bench) (provides the server source code) and start the servers:

```bash
git clone https://github.com/Accenture/mcp-bench.git ../mcp-bench
bash start_servers.sh          # installs deps + starts 6 servers on ports 8001-8006
bash start_servers.sh --check  # verify they're running
```

**Langflow agents**: Each MCP server needs a Langflow agent flow connected to it for the task generation step (Section 2).

1. Start Langflow: `uvx langflow run`
2. For each server, create a flow with **Agent** + **MCP Tools**:
   - Point MCP Tools at the server URL (e.g., `http://localhost:8001/mcp`)
   - Set Agent **Max Iterations = 100**
   - Configure the Agent LLM (e.g., GPT-5.2)
3. Note each flow's URL and add to `.env`

**Note**: The exploration step may occasionally fail if the frontier model probes edge cases
that cause MCP tool errors (e.g., empty arrays, division by zero). The distillation flow has
built-in quality filtering that handles this — simply re-run the cell if a server fails.
Pre-generated tasks are provided in `outputs/` so you can skip Section 2 entirely if needed.

**Environment**: Copy `.env.example` to `.env` and fill in your API key + Langflow URLs.

In [ ]:
import os
import sys
import json
import asyncio
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

import nest_asyncio
nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(NOTEBOOK_DIR / ".env")

# LLM API key used as the default for all OpenAI model calls
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
assert OPENAI_API_KEY and OPENAI_API_KEY != "sk-...", "Set OPENAI_API_KEY in .env"

# Teacher model: used by the distillation flow for question generation + quality scoring (Section 2)
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "openai/gpt-5.2")

# Judge model: LLM-as-judge that scores each evaluated model's trace against the expert gold standard (Section 4)
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "openai/gpt-4o")

# Langflow API key: optional authentication for Langflow agent flows (Section 2)
LANGFLOW_API_KEY = os.environ.get("LANGFLOW_API_KEY", None)

# ── MCP Server Ports ──────────────────────────────────────────────────
# Configure the port for each MCP server. Default: 8001-8006.
# Each server runs via supergateway on its assigned port.
SERVER_PORTS = {
    "Weather Data": 8001,
    "Medical Calculator": 8002,
    "Wikipedia": 8003,
    "Car Price Evaluator": 8004,
    "Reddit": 8005,
    "DEX Paprika": 8006,
}

# Server -> MCP URL (built from ports)
MCP_SERVERS = {
    name: f"http://localhost:{port}/mcp"
    for name, port in SERVER_PORTS.items()
}

# Server -> Langflow agent URL (each flow has a frontier model agent connected to one MCP server)
LANGFLOW_URLS = {
    "Weather Data": os.environ.get("LANGFLOW_URL_WEATHER_DATA", ""),
    "Medical Calculator": os.environ.get("LANGFLOW_URL_MEDICAL_CALCULATOR", ""),
    "Wikipedia": os.environ.get("LANGFLOW_URL_WIKIPEDIA", ""),
    "Car Price Evaluator": os.environ.get("LANGFLOW_URL_CAR_PRICE", ""),
    "Reddit": os.environ.get("LANGFLOW_URL_REDDIT", ""),
    "DEX Paprika": os.environ.get("LANGFLOW_URL_DEX_PAPRIKA", ""),
}

# ── Models to evaluate ────────────────────────────────────────────────
# Each model can have optional overrides for api_key and api_base.
# Default: uses OPENAI_API_KEY. For local/custom models, set api_base.
#
# Examples:
#   "openai/gpt-4o": {}                                               # OpenAI API
#   "hosted_vllm/my-local-model": {"api_base": "http://localhost:8000/v1"}  # vLLM
#   "vertex_ai/claude-sonnet-4@20250514": {"api_key": None}           # Vertex AI (ADC)
#
MODEL_CONFIGS = {
    "openai/gpt-5": {},
    "openai/gpt-4o": {},
    "openai/gpt-4o-mini": {},
}

EVAL_MODELS = list(MODEL_CONFIGS.keys())

print(f"Teacher: {TEACHER_MODEL}")
print(f"Judge:   {JUDGE_MODEL}")
print(f"Models:  {EVAL_MODELS}")
print(f"Servers: {list(MCP_SERVERS.keys())}")
print(f"Ports:   {list(SERVER_PORTS.values())}")

---
## 1. Discover Tools

Connect to each MCP server and list its tools.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def discover_tools(url):
    async with streamablehttp_client(url) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resp = await session.list_tools()
            return [{"name": t.name, "description": t.description or "", "inputSchema": t.inputSchema} for t in resp.tools]

all_tools = {}
for name, url in MCP_SERVERS.items():
    try:
        tools = asyncio.run(discover_tools(url))
        all_tools[name] = tools
        print(f"  {name}: {len(tools)} tools")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

assert all_tools, "No servers reachable! Start them with: bash start_servers.sh"
print(f"\nTotal: {sum(len(t) for t in all_tools.values())} tools across {len(all_tools)} servers")

---
## 2. Generate Synthetic Tasks

Run the MCP distillation flow on each server at varying complexity levels.

The `num_samples` parameter controls how many tools each generated question is designed around:
- `num_samples=2` → simpler questions using 2 tools
- `num_samples=4` → moderate questions using 4 tools
- `num_samples=8` → complex questions using 8 tools

Servers with fewer tools than `num_samples` are automatically skipped for that level.
Tasks from all levels are combined into a single file per server.

In [ ]:
from sdg_hub import Flow, FlowRegistry

FlowRegistry.discover_flows()

# ── Configuration ─────────────────────────────────────────────────────
# Tool complexity levels to generate tasks at.
# Each level samples N tools per question from the server's tool set.
# Servers with fewer tools than N are skipped for that level.
NUM_SAMPLES_LEVELS = [2, 4, 8]

# Server descriptions (used in the generation prompt)
SERVER_DESCRIPTIONS = {
    "Weather Data": "Weather data server providing current and forecast weather information.",
    "Medical Calculator": "Medical calculator server with 22 clinical calculation tools.",
    "Wikipedia": "Wikipedia server providing article search, content, and summarization.",
    "Car Price Evaluator": "Vehicle market pricing server for Brazilian car brands.",
    "Reddit": "Reddit server for fetching hot threads and post content.",
    "DEX Paprika": "DeFi/crypto analytics server with pool, token, and network data.",
}


def generate_tasks_for_server(server_name, tools, langflow_url, num_samples=2):
    """Run the distillation flow for one server at a given num_samples level."""
    flow_instance = Flow.from_yaml(FlowRegistry.get_flow_path("MCP Server Distillation"))
    flow_instance.set_model_config(model=TEACHER_MODEL, api_key=OPENAI_API_KEY)

    agent_kwargs = {"agent_framework": "langflow", "agent_url": langflow_url}
    if LANGFLOW_API_KEY:
        agent_kwargs["agent_api_key"] = LANGFLOW_API_KEY
    flow_instance.set_agent_config(**agent_kwargs)
    flow_instance.set_agent_config(timeout=300, blocks=["explore_server"])

    df = pd.DataFrame({
        "tool_list": [tools],
        "mcp_server_name": [server_name],
        "mcp_server_description": [SERVER_DESCRIPTIONS.get(server_name, f"{server_name} MCP server")],
    })

    runtime_params = {}
    if num_samples != 2:
        runtime_params["sample_tools"] = {"num_samples": num_samples}

    result = flow_instance.generate(df, runtime_params=runtime_params)
    result_df = result.to_pandas() if hasattr(result, "to_pandas") else result

    export_cols = [c for c in [
        "question", "target_tools", "extract_agent_text_text",
        "extract_agent_text_tool_trace", "question_quality_rating",
        "completeness_rating", "tool_list",
    ] if c in result_df.columns]
    return result_df[export_cols]


print(f"Flow loaded: MCP Server Distillation")
print(f"Complexity levels: {NUM_SAMPLES_LEVELS}")
print(f"Servers: {list(all_tools.keys())}")

In [ ]:
# Generate tasks across all servers and complexity levels
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

for server_name, tools in all_tools.items():
    safe_name = server_name.replace(" ", "_")
    out_path = OUTPUT_DIR / f"{safe_name}.jsonl"

    # Skip if already generated
    if out_path.exists():
        with open(out_path) as fp:
            n = sum(1 for _ in fp)
        print(f"{server_name}: {n} tasks (cached — delete {out_path.name} to regenerate)")
        continue

    langflow_url = LANGFLOW_URLS.get(server_name, "")
    if not langflow_url:
        print(f"{server_name}: SKIP — no Langflow URL in .env")
        continue

    all_tasks = []
    n_tools = len(tools)

    for ns in NUM_SAMPLES_LEVELS:
        if n_tools < ns:
            print(f"  {server_name} ns={ns}: skipped ({n_tools} tools < {ns})")
            continue

        print(f"\n{'─' * 50}")
        print(f"{server_name} — num_samples={ns} ({n_tools} tools)")
        print(f"{'─' * 50}")

        try:
            result_df = generate_tasks_for_server(server_name, tools, langflow_url, num_samples=ns)
            all_tasks.append(result_df)
            print(f"  Generated {len(result_df)} tasks")
        except Exception as e:
            print(f"  FAILED: {e}")

    if all_tasks:
        combined = pd.concat(all_tasks, ignore_index=True)
        combined.to_json(out_path, orient="records", lines=True)
        print(f"\n  {server_name}: saved {len(combined)} total tasks → {out_path.name}")
    else:
        print(f"\n  {server_name}: no tasks generated")

In [ ]:
# Summary of generated tasks
print("Generated Tasks:")
print("─" * 50)
total = 0
for f in sorted(OUTPUT_DIR.glob("*.jsonl")):
    server = f.stem.replace("_", " ")
    with open(f) as fp:
        n = sum(1 for _ in fp)
    total += n
    print(f"  {server:<25} {n} tasks")
print(f"  {'TOTAL':<25} {total} tasks")

---
## 3. Inspect Generated Tasks

Let's look at a sample task to understand what the distillation flow produces.

In [ ]:
# Pick a sample task
sample_file = sorted(OUTPUT_DIR.glob("*.jsonl"))[0]
with open(sample_file) as f:
    sample = json.loads(f.readline())

server = sample_file.stem.replace("_", " ")
print(f"Server: {server}")
print(f"\nQuestion:")
print(f"  {sample['question'][:400]}")
print(f"\nTarget Tools:")
print(f"  {sample['target_tools']}")
print(f"\nQuality: {sample.get('question_quality_rating', '?')}")
print(f"Completeness: {sample.get('completeness_rating', '?')}")

# Expert trajectory
trace = sample.get("extract_agent_text_tool_trace", [])
if isinstance(trace, str):
    trace = json.loads(trace)

tool_calls = [s for s in trace if isinstance(s, dict) and s.get("type") == "tool_use"]
print(f"\nExpert Trajectory: {len(tool_calls)} tool calls")
for i, tc in enumerate(tool_calls):
    print(f"  [{i+1}] {tc['name']}({json.dumps(tc.get('tool_input', {}))[:80]})")

print(f"\nExpert Answer (first 300 chars):")
print(f"  {sample.get('extract_agent_text_text', '')[:300]}")

---
## 4. Evaluate Models

Run each model on the generated questions via MCPAgentBlock (direct MCP connection,
no Langflow needed), then score against the expert gold standard.

In [ ]:
from sdg_hub.core.blocks import MCPAgentBlock
from pydantic import SecretStr
from litellm import acompletion


def extract_tools_from_trace(trace):
    """Extract tool names from MCPAgentBlock trace."""
    tools = []
    for msg in trace.get("messages", []):
        if msg.get("role") == "assistant" and msg.get("tool_calls"):
            for tc in msg["tool_calls"]:
                fn = tc.get("function", tc)
                if fn.get("name"):
                    tools.append(fn["name"])
    return tools


def extract_expert_tools(expert_trace):
    """Extract tool names from distillation flow expert trace."""
    if isinstance(expert_trace, str):
        expert_trace = json.loads(expert_trace)
    return [s["name"] for s in expert_trace if isinstance(s, dict) and s.get("type") == "tool_use"]


def extract_final_answer(trace):
    for msg in reversed(trace.get("messages", [])):
        if msg.get("role") == "assistant" and not msg.get("tool_calls"):
            return msg.get("content", "") or ""
    return ""


def compute_tool_metrics(model_tools, expert_tools):
    model_set, expert_set = set(model_tools), set(expert_tools)
    if not expert_set:
        return {"tool_recall": 1.0, "tool_precision": 1.0, "order_match": 1.0}
    intersection = model_set & expert_set
    recall = len(intersection) / len(expert_set)
    precision = len(intersection) / len(model_set) if model_set else 0.0
    # LCS for order match
    m, n = len(model_tools), len(expert_tools)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if model_tools[i-1] == expert_tools[j-1] else max(dp[i-1][j], dp[i][j-1])
    order = dp[m][n] / len(expert_tools)
    return {"tool_recall": round(recall, 3), "tool_precision": round(precision, 3), "order_match": round(order, 3)}


async def judge_trace(question, model_answer, expert_answer, model_tools, expert_tools):
    prompt = f"""You are evaluating an AI agent's performance on a tool-use task.

## Question\n{question}
## Expert's Answer\n{expert_answer[:2000]}
## Expert's Tools\n{', '.join(expert_tools)}
## Model's Answer\n{model_answer[:2000]}
## Model's Tools\n{', '.join(model_tools)}

Score on three dimensions (0-10 each):
1. **task_completion**: Did the model achieve the same objectives as the expert?
2. **tool_usage**: Did the model use appropriate tools with correct parameters?
3. **answer_quality**: Is the final answer as complete and accurate as the expert's?

Respond with JSON only:
```json
{{"task_completion": 7, "tool_usage": 8, "answer_quality": 6, "rationale": "Brief explanation."}}
```"""
    response = await acompletion(
        model=JUDGE_MODEL, api_key=OPENAI_API_KEY, temperature=0.0, max_tokens=512,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": "Respond with JSON only."}, {"role": "user", "content": prompt}],
    )
    return json.loads(response.choices[0].message.content)


print("Evaluation functions loaded.")

In [ ]:
# Run evaluation: for each server x each model
# Results are cached — if evaluation_results.jsonl exists, loads from cache.
# Delete the file to force re-evaluation.

RESULTS_PATH = NOTEBOOK_DIR / "evaluation_results.jsonl"
EVAL_MODELS_TO_RUN = []

if RESULTS_PATH.exists():
    all_results_df = pd.read_json(RESULTS_PATH, orient="records", lines=True)

    cached_models = set(all_results_df["model"].unique())
    requested_models = set(EVAL_MODELS)
    missing_models = requested_models - cached_models

    if not missing_models:
        all_results = all_results_df.to_dict("records")
        print(f"Loaded {len(all_results)} cached results from {RESULTS_PATH.name}")
        print(f"Models: {sorted(cached_models)}")
        print(f"Servers: {sorted(all_results_df['server'].unique().tolist())}")
    else:
        print(f"Cache exists but missing models: {missing_models}")
        print(f"Running evaluation for missing models...")
        all_results = all_results_df.to_dict("records")
        EVAL_MODELS_TO_RUN = list(missing_models)
else:
    all_results = []
    EVAL_MODELS_TO_RUN = list(EVAL_MODELS)

# Run evaluation for any uncached models
if EVAL_MODELS_TO_RUN:
    judge_failures = 0

    for server_name in MCP_SERVERS:
        safe_name = server_name.replace(" ", "_")
        tasks_path = OUTPUT_DIR / f"{safe_name}.jsonl"
        if not tasks_path.exists():
            print(f"SKIP {server_name}: no tasks")
            continue

        tasks_df = pd.read_json(tasks_path, orient="records", lines=True)
        server_url = MCP_SERVERS[server_name]

        print(f"\n{'=' * 60}")
        print(f"{server_name} ({len(tasks_df)} tasks)")
        print(f"{'=' * 60}")

        for model in EVAL_MODELS_TO_RUN:
            model_short = model.split("/")[-1]
            config = MODEL_CONFIGS.get(model, {})

            model_api_key = config.get("api_key", OPENAI_API_KEY)
            model_api_base = config.get("api_base", None)

            print(f"\n  {model_short}:", end=" ")

            block_kwargs = {
                "block_name": f"eval_{model_short}_{safe_name}",
                "mcp_server_url": server_url,
                "model": model,
                "max_iterations": 20,
                "input_cols": ["question"],
                "output_cols": ["model_trace"],
            }
            if model_api_key is not None:
                block_kwargs["api_key"] = SecretStr(model_api_key)
            if model_api_base is not None:
                block_kwargs["api_base"] = model_api_base

            block = MCPAgentBlock(**block_kwargs)

            try:
                result_df = block.generate(tasks_df[["question"]].copy())
            except Exception as e:
                print(f"FAILED ({e})")
                continue

            for idx in range(len(result_df)):
                trace = result_df["model_trace"].iloc[idx]
                expert_trace = tasks_df["extract_agent_text_tool_trace"].iloc[idx]
                expert_answer = tasks_df["extract_agent_text_text"].iloc[idx]
                question = tasks_df["question"].iloc[idx]

                model_tools = extract_tools_from_trace(trace)
                exp_tools = extract_expert_tools(expert_trace)
                tool_metrics = compute_tool_metrics(model_tools, exp_tools)
                model_answer = extract_final_answer(trace)

                try:
                    judge = asyncio.run(judge_trace(question, model_answer, expert_answer, model_tools, exp_tools))
                except Exception as e:
                    print(f"\n    Judge failed for task {idx}: {e}")
                    judge = {"task_completion": 0, "tool_usage": 0, "answer_quality": 0}
                    judge_failures += 1

                all_results.append({"server": server_name, "model": model, "task_idx": idx, **tool_metrics, **judge})

            scores = [r for r in all_results if r["model"] == model and r["server"] == server_name]
            avg = np.mean([s.get("task_completion", 0) for s in scores])
            print(f"{len(scores)} tasks, avg_completion={avg:.1f}")

    if judge_failures:
        print(f"\nWARNING: {judge_failures} judge call(s) failed — those tasks scored 0.")

    # Save combined results
    pd.DataFrame(all_results).to_json(RESULTS_PATH, orient="records", lines=True)
    print(f"\nSaved {len(all_results)} results to {RESULTS_PATH.name}")

print(f"\nTotal: {len(all_results)} evaluation scores")

---
## 5. Results

In [ ]:
results_df = pd.DataFrame(all_results)

judge_cols = ["task_completion", "tool_usage", "answer_quality"]
tool_cols = ["tool_recall", "tool_precision", "order_match"]
all_cols = tool_cols + judge_cols

# Per server x model table
scores = results_df.groupby(["server", "model"])[all_cols].mean()
for col in judge_cols:
    scores[col] = scores[col] / 10.0
scores["overall"] = scores.mean(axis=1)

pivot = scores["overall"].unstack("model")
pivot.columns = [c.split("/")[-1] for c in pivot.columns]
col_order = pivot.mean().sort_values(ascending=False).index.tolist()
pivot = pivot[col_order]

# Task count: total rows per server divided by number of models
n_models = results_df["model"].nunique()
task_counts = results_df.groupby("server").size() // n_models
pivot.insert(0, "tasks", task_counts)

# Overall row
short_to_full = {m.split("/")[-1]: m for m in EVAL_MODELS}
pivot.loc["OVERALL"] = None
overall = results_df.groupby("model")[all_cols].mean()
for col in judge_cols:
    overall[col] = overall[col] / 10.0
overall["overall"] = overall.mean(axis=1)
for model_short in col_order:
    full_name = short_to_full.get(model_short, model_short)
    if full_name in overall.index:
        pivot.loc["OVERALL", model_short] = overall.loc[full_name, "overall"]
pivot.loc["OVERALL", "tasks"] = len(results_df) // n_models

server_rows = pivot.drop("OVERALL").sort_values(col_order[0], ascending=False)
pivot = pd.concat([server_rows, pivot.loc[["OVERALL"]]])

print("Overall Score by Server x Model")
print("─" * 60)
print(pivot.round(3).to_string())

ranking = sorted(col_order, key=lambda m: pivot.loc["OVERALL", m], reverse=True)
print(f"\nRanking: {' > '.join(ranking)}")

# Save results
RESULTS_PATH = NOTEBOOK_DIR / "evaluation_results.jsonl"
results_df.to_json(RESULTS_PATH, orient="records", lines=True)
print(f"\nSaved {len(results_df)} evaluation scores to {RESULTS_PATH.name}")